# Step 3: Double Buffering + Float4 Store

**Runtime -> Change runtime type -> GPU (T4)**

## 优化内容

### 1. Double Buffering（主要优化）

**原理**: 分配两份 shared memory buffer，当线程计算当前 tile 时，同时预取下一个 tile 到另一份 buffer。

原版每个 tile 需要 2 次 `__syncthreads()`：
```
load tile -> sync -> compute tile -> sync
```

Double buffering 只需 1 次（首个 tile 除外）：
```
prefetch tile[0] -> sync
loop:
    load tile[k+1] -> buf[write]    (与下面的 compute 在 warp 级别重叠)
    compute tile[k] <- buf[read]
    swap buffers
    sync
compute last tile
```

**代价**: shared memory 翻倍（8.7KB -> 17.4KB），需要设置 `cudaFuncCachePreferShared` 确保 T4 配置 48KB shared memory，否则每 SM 只能放 1 个 block（occupancy 降至 25%）。

### 2. Float4 Vectorized Store（次要优化）

**问题**: 写回 C 时每次只写 1 个 float，同一 warp 的 16 个 tx 线程写入列间距 8 -> 每个 128B 事务只用 4B (12.5%)。

**修复**: 用 `float4` 一次写 4 个连续 float (16B)，每个事务利用率从 12.5% -> 50%。

### 3. 保留 v3 的 bank conflict 修复

As PAD=1 + Bs XOR swizzle 继续使用。

In [1]:
# 检查 GPU 和 CUDA 版本
!nvidia-smi
!nvcc --version

Mon Apr  6 03:06:29 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   31C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
%%writefile matmul_reg_tiling_v4.cu
#include <stdio.h>
#include <stdlib.h>
#include <math.h>
#include <cuda_runtime.h>

#define BM 128
#define BN 128
#define BK 8
#define TM 8
#define TN 8
#define A_PAD 1
#define BS_SWIZZLE(col) ((col) ^ (((col) >> 3) << 1))

// ============================================================
// v4 kernel: Double Buffering + Float4 Store
// ============================================================
// 在 v3 (As PAD + Bs swizzle) 基础上增加:
//   1. Double buffering: 2 份 shared memory, 加载和计算重叠
//   2. Float4 store: 写回 C 时用 128bit 写入, 提高 global store 效率
//
// Shared memory 用量: 2 * (As[128][9] + Bs[8][128]) = 17.4 KB
// 需要 cudaFuncCachePreferShared 配置 48KB shared memory
// 否则 2 blocks * 17.4KB = 34.8KB > 默认 32KB, occupancy 会降半
// ============================================================
__global__ void MatMulRegTilingKernel_v4(const float* __restrict__ A,
                                         const float* __restrict__ B,
                                         float* __restrict__ C,
                                         int M, int N, int K)
{
    const int bx = blockIdx.x;
    const int by = blockIdx.y;
    const int tx = threadIdx.x;
    const int ty = threadIdx.y;
    const int tid = ty * blockDim.x + tx;
    const int numThreads = blockDim.x * blockDim.y;

    // Double buffer: 两份 shared memory
    __shared__ float As[2][BM][BK + A_PAD];  // 2 x 128 x 9
    __shared__ float Bs[2][BK][BN];          // 2 x 8 x 128

    float regC[TM][TN] = {0.0f};
    float regA[TM];
    float regB[TN];

    const int cRow = by * BM;
    const int cCol = bx * BN;
    const float* aPtr = A + cRow * K;
    const float* bPtr = B + cCol;

    const int loadARow = tid / BK;
    const int loadACol = tid % BK;
    const int strideA  = numThreads / BK;
    const int loadBRow = tid / BN;
    const int loadBCol = tid % BN;
    const int strideB  = numThreads / BN;

    // ========== 预取第一个 tile 到 buffer 0 ==========
    for (int i = 0; i < BM; i += strideA) {
        As[0][loadARow + i][loadACol] = aPtr[(loadARow + i) * K + loadACol];
    }
    for (int i = 0; i < BK; i += strideB) {
        Bs[0][loadBRow + i][BS_SWIZZLE(loadBCol)] = bPtr[(loadBRow + i) * N + loadBCol];
    }
    __syncthreads();

    // ========== 主循环: double buffering ==========
    // writeBuffer: 下一个 tile 写入的 buffer
    // readBuffer:  当前 tile 读取的 buffer (= 1 - writeBuffer)
    int writeBuffer = 1;
    for (int k = BK; k < K; k += BK) {
        int readBuffer = 1 - writeBuffer;

        // 加载下一个 tile 到 writeBuffer
        // (与下面的 compute 操作访问不同 buffer, 可在 warp 级别重叠)
        for (int i = 0; i < BM; i += strideA) {
            As[writeBuffer][loadARow + i][loadACol] =
                aPtr[(loadARow + i) * K + k + loadACol];
        }
        for (int i = 0; i < BK; i += strideB) {
            Bs[writeBuffer][loadBRow + i][BS_SWIZZLE(loadBCol)] =
                bPtr[(k + loadBRow + i) * N + loadBCol];
        }

        // 从 readBuffer 计算当前 tile
        for (int bk = 0; bk < BK; ++bk) {
            for (int tm = 0; tm < TM; ++tm)
                regA[tm] = As[readBuffer][ty * TM + tm][bk];
            for (int tn = 0; tn < TN; ++tn)
                regB[tn] = Bs[readBuffer][bk][BS_SWIZZLE(tx * TN + tn)];
            for (int tm = 0; tm < TM; ++tm)
                for (int tn = 0; tn < TN; ++tn)
                    regC[tm][tn] += regA[tm] * regB[tn];
        }

        writeBuffer = 1 - writeBuffer;
        __syncthreads();  // 确保 load 和 compute 都完成后再交换
    }

    // ========== 计算最后一个 tile ==========
    {
        int readBuffer = 1 - writeBuffer;
        for (int bk = 0; bk < BK; ++bk) {
            for (int tm = 0; tm < TM; ++tm)
                regA[tm] = As[readBuffer][ty * TM + tm][bk];
            for (int tn = 0; tn < TN; ++tn)
                regB[tn] = Bs[readBuffer][bk][BS_SWIZZLE(tx * TN + tn)];
            for (int tm = 0; tm < TM; ++tm)
                for (int tn = 0; tn < TN; ++tn)
                    regC[tm][tn] += regA[tm] * regB[tn];
        }
    }

    // ========== Float4 向量化写回 C ==========
    // 每次写 4 个连续 float (128bit), 减少 global store 事务数
    // 地址对齐: cCol 是 128 的倍数, tx*TN 是 8 的倍数 -> 32B 对齐 > 16B 要求
    for (int tm = 0; tm < TM; ++tm) {
        int globalRow = cRow + ty * TM + tm;
        reinterpret_cast<float4*>(&C[globalRow * N + cCol + tx * TN])[0] =
            make_float4(regC[tm][0], regC[tm][1], regC[tm][2], regC[tm][3]);
        reinterpret_cast<float4*>(&C[globalRow * N + cCol + tx * TN + 4])[0] =
            make_float4(regC[tm][4], regC[tm][5], regC[tm][6], regC[tm][7]);
    }
}

// ============================================================
// v3 kernel: 用于对比 (As PAD + Bs swizzle, 无 double buffer)
// ============================================================
__global__ void MatMulRegTilingKernel_v3(const float* __restrict__ A,
                                         const float* __restrict__ B,
                                         float* __restrict__ C,
                                         int M, int N, int K)
{
    const int bx = blockIdx.x;
    const int by = blockIdx.y;
    const int tx = threadIdx.x;
    const int ty = threadIdx.y;
    const int tid = ty * blockDim.x + tx;
    const int numThreads = blockDim.x * blockDim.y;

    __shared__ float As[BM][BK + A_PAD];
    __shared__ float Bs[BK][BN];

    float regC[TM][TN] = {0.0f};
    float regA[TM];
    float regB[TN];

    const int cRow = by * BM;
    const int cCol = bx * BN;
    const float* aPtr = A + cRow * K;
    const float* bPtr = B + cCol;

    const int loadARow = tid / BK;
    const int loadACol = tid % BK;
    const int strideA  = numThreads / BK;
    const int loadBRow = tid / BN;
    const int loadBCol = tid % BN;
    const int strideB  = numThreads / BN;

    for (int k = 0; k < K; k += BK) {
        for (int i = 0; i < BM; i += strideA)
            As[loadARow + i][loadACol] = aPtr[(loadARow + i) * K + k + loadACol];
        for (int i = 0; i < BK; i += strideB)
            Bs[loadBRow + i][BS_SWIZZLE(loadBCol)] = bPtr[(k + loadBRow + i) * N + loadBCol];
        __syncthreads();

        for (int bk = 0; bk < BK; ++bk) {
            for (int tm = 0; tm < TM; ++tm)
                regA[tm] = As[ty * TM + tm][bk];
            for (int tn = 0; tn < TN; ++tn)
                regB[tn] = Bs[bk][BS_SWIZZLE(tx * TN + tn)];
            for (int tm = 0; tm < TM; ++tm)
                for (int tn = 0; tn < TN; ++tn)
                    regC[tm][tn] += regA[tm] * regB[tn];
        }
        __syncthreads();
    }

    for (int tm = 0; tm < TM; ++tm)
        for (int tn = 0; tn < TN; ++tn) {
            int globalRow = cRow + ty * TM + tm;
            int globalCol = cCol + tx * TN + tn;
            C[globalRow * N + globalCol] = regC[tm][tn];
        }
}

int main()
{
    int M = 2048, N = 2048, K = 2048;

    size_t bytesA = M * K * sizeof(float);
    size_t bytesB = K * N * sizeof(float);
    size_t bytesC = M * N * sizeof(float);

    float* A     = (float*)malloc(bytesA);
    float* B     = (float*)malloc(bytesB);
    float* C_v3  = (float*)malloc(bytesC);
    float* C_v4  = (float*)malloc(bytesC);

    for (int i = 0; i < M * K; ++i) A[i] = (float)(i % 10) * 0.1f;
    for (int i = 0; i < K * N; ++i) B[i] = (float)(i % 7)  * 0.1f;

    float *d_A, *d_B, *d_C;
    cudaMalloc(&d_A, bytesA);
    cudaMalloc(&d_B, bytesB);
    cudaMalloc(&d_C, bytesC);
    cudaMemcpy(d_A, A, bytesA, cudaMemcpyHostToDevice);
    cudaMemcpy(d_B, B, bytesB, cudaMemcpyHostToDevice);

    dim3 dimBlock(BN / TN, BM / TM);
    dim3 dimGrid(N / BN, M / BM);

    // v4 需要更多 shared memory, 设置 prefer shared
    // T4 默认 32KB shared, double buffer 需要 17.4KB * 2 blocks = 34.8KB
    // PreferShared 配置为 48KB shared + 16KB L1
    cudaFuncSetCacheConfig(MatMulRegTilingKernel_v4, cudaFuncCachePreferShared);

    cudaEvent_t start, stop;
    cudaEventCreate(&start);
    cudaEventCreate(&stop);
    float ms = 0;

    // ========== v3 baseline ==========
    MatMulRegTilingKernel_v3<<<dimGrid, dimBlock>>>(d_A, d_B, d_C, M, N, K);
    cudaDeviceSynchronize();
    cudaEventRecord(start);
    MatMulRegTilingKernel_v3<<<dimGrid, dimBlock>>>(d_A, d_B, d_C, M, N, K);
    cudaEventRecord(stop);
    cudaEventSynchronize(stop);
    cudaEventElapsedTime(&ms, start, stop);
    float v3_ms = ms;
    cudaMemcpy(C_v3, d_C, bytesC, cudaMemcpyDeviceToHost);

    // ========== v4 double buffer + float4 store ==========
    MatMulRegTilingKernel_v4<<<dimGrid, dimBlock>>>(d_A, d_B, d_C, M, N, K);
    cudaDeviceSynchronize();
    cudaEventRecord(start);
    MatMulRegTilingKernel_v4<<<dimGrid, dimBlock>>>(d_A, d_B, d_C, M, N, K);
    cudaEventRecord(stop);
    cudaEventSynchronize(stop);
    cudaEventElapsedTime(&ms, start, stop);
    float v4_ms = ms;
    cudaMemcpy(C_v4, d_C, bytesC, cudaMemcpyDeviceToHost);

    // ========== 验证 ==========
    float max_err = 0.0f;
    for (int i = 0; i < M * N; ++i) {
        float err = fabsf(C_v3[i] - C_v4[i]);
        if (err > max_err) max_err = err;
    }

    double gflops_v3 = 2.0 * M * N * K / (v3_ms * 1e-3) / 1e9;
    double gflops_v4 = 2.0 * M * N * K / (v4_ms * 1e-3) / 1e9;

    printf("=== Register Tiling GEMM: v3 vs v4 ===\n");
    printf("Matrix: %dx%dx%d\n", M, N, K);
    printf("Block tile: %dx%d, Thread tile: %dx%d, K tile: %d\n", BM, BN, TM, TN, BK);
    printf("\n");
    printf("v3 (PAD+swizzle):           %.3f ms  (%.1f GFLOPS)\n", v3_ms, gflops_v3);
    printf("v4 (+ double buf + f4 st):  %.3f ms  (%.1f GFLOPS)\n", v4_ms, gflops_v4);
    printf("Speedup v4/v3: %.2fx\n", v3_ms / v4_ms);
    printf("Max diff (v3 vs v4): %e\n", max_err);
    if (max_err < 1e-3f)
        printf("PASSED\n");
    else
        printf("FAILED\n");

    cudaEventDestroy(start); cudaEventDestroy(stop);
    cudaFree(d_A); cudaFree(d_B); cudaFree(d_C);
    free(A); free(B); free(C_v3); free(C_v4);
    return 0;
}

Writing matmul_reg_tiling_v4.cu


In [3]:
!nvcc -O3 -lineinfo -arch=sm_75 matmul_reg_tiling_v4.cu -o matmul_reg_tiling_v4
print("编译成功!")

编译成功!


In [4]:
# 运行对比测试
!./matmul_reg_tiling_v4

=== Register Tiling GEMM: v3 vs v4 ===
Matrix: 2048x2048x2048
Block tile: 128x128, Thread tile: 8x8, K tile: 8

v3 (PAD+swizzle):           6.242 ms  (2752.2 GFLOPS)
v4 (+ double buf + f4 st):  6.050 ms  (2839.9 GFLOPS)
Speedup v4/v3: 1.03x
Max diff (v3 vs v4): 0.000000e+00
PASSED


---
## Part 1: Nsight Systems

In [5]:
from google.colab import drive
drive.mount('/content/drive')
!chmod -R +x /content/drive/MyDrive/cuda/nsys_tool

NSYS = "/content/drive/MyDrive/cuda/nsys_tool/target-linux-x64/nsys"
!{NSYS} --version

Mounted at /content/drive
NVIDIA Nsight Systems version 2024.5.1.113-245134619542v0


In [6]:
NSYS = "/content/drive/MyDrive/cuda/nsys_tool/target-linux-x64/nsys"
!{NSYS} profile \
     --stats=true \
     --force-overwrite=true \
     -o nsys_reg_tiling_v4 \
     ./matmul_reg_tiling_v4

=== Register Tiling GEMM: v3 vs v4 ===
Matrix: 2048x2048x2048
Block tile: 128x128, Thread tile: 8x8, K tile: 8

v3 (PAD+swizzle):           9.453 ms  (1817.4 GFLOPS)
v4 (+ double buf + f4 st):  9.110 ms  (1885.8 GFLOPS)
Speedup v4/v3: 1.04x
Max diff (v3 vs v4): 0.000000e+00
PASSED
Generating '/tmp/nsys-report-99b3.qdstrm'
[1/8] [========================100%] nsys_reg_tiling_v4.nsys-rep
[2/8] [========================100%] nsys_reg_tiling_v4.sqlite
[3/8] Executing 'nvtx_sum' stats report
SKIPPED: /content/nsys_reg_tiling_v4.sqlite does not contain NV Tools Extension (NVTX) data.
[4/8] Executing 'osrt_sum' stats report

 Time (%)  Total Time (ns)  Num Calls    Avg (ns)      Med (ns)    Min (ns)   Max (ns)     StdDev (ns)            Name         
 --------  ---------------  ---------  ------------  ------------  --------  -----------  -------------  ----------------------
     74.9      829,528,255         15  55,301,883.7  17,874,240.0     1,840  528,955,438  134,140,288.1  poll         

In [7]:
NSYS = "/content/drive/MyDrive/cuda/nsys_tool/target-linux-x64/nsys"
!{NSYS} stats --force-export=true --report cuda_gpu_kern_sum nsys_reg_tiling_v4.nsys-rep

Generating SQLite file nsys_reg_tiling_v4.sqlite from nsys_reg_tiling_v4.nsys-rep
Processing [nsys_reg_tiling_v4.sqlite] with [/content/drive/MyDrive/cuda/nsys_tool/host-linux-x64/reports/cuda_gpu_kern_sum.py]... 

 ** CUDA GPU Kernel Summary (cuda_gpu_kern_sum):

 Time (%)  Total Time (ns)  Instances   Avg (ns)     Med (ns)    Min (ns)   Max (ns)   StdDev (ns)                                       Name                                     
 --------  ---------------  ---------  -----------  -----------  ---------  ---------  -----------  ------------------------------------------------------------------------------
     50.8       18,744,124          2  9,372,062.0  9,372,062.0  9,367,934  9,376,190      5,837.9  MatMulRegTilingKernel_v3(const float *, const float *, float *, int, int, int)
     49.2       18,182,122          2  9,091,061.0  9,091,061.0  9,087,685  9,094,437      4,774.4  MatMulRegTilingKernel_v4(const float *, const float *, float *, int, int, int)



---
## Part 2: Nsight Compute — v4 深度分析

重点验证:
1. double buffering 是否降低了 warp stall / 提高了 IPC
2. float4 store 是否改善了 global store coalescing
3. shared memory 翻倍后 occupancy 是否受影响

In [8]:
# Memory Workload
!ncu --section MemoryWorkloadAnalysis \
     --section MemoryWorkloadAnalysis_Tables \
     --kernel-name MatMulRegTilingKernel_v4 \
     --launch-skip 1 --launch-count 1 \
     ./matmul_reg_tiling_v4

==PROF== Connected to process 9292 (/content/matmul_reg_tiling_v4)
==PROF== Profiling "MatMulRegTilingKernel_v4": 0%....50%....100% - 27 passes
=== Register Tiling GEMM: v3 vs v4 ===
Matrix: 2048x2048x2048
Block tile: 128x128, Thread tile: 8x8, K tile: 8

v3 (PAD+swizzle):           8.038 ms  (2137.2 GFLOPS)
v4 (+ double buf + f4 st):  5988.255 ms  (2.9 GFLOPS)
Speedup v4/v3: 0.00x
Max diff (v3 vs v4): 0.000000e+00
PASSED
==PROF== Disconnected from process 9292
[9292] matmul_reg_tiling_v4@127.0.0.1
  MatMulRegTilingKernel_v4(const float *, const float *, float *, int, int, int) (16, 16, 1)x(16, 16, 1), Context 1, Stream 7, Device 0, CC 7.5
    Section: Memory Workload Analysis
    ----------------- ----------- ------------
    Metric Name       Metric Unit Metric Value
    ----------------- ----------- ------------
    Memory Throughput     Gbyte/s        22.61
    Mem Busy                    %        37.39
    Max Bandwidth               %        66.06
    L1/TEX Hit Rate             

In [9]:
# Occupancy + Launch Stats
!ncu --section Occupancy \
     --section LaunchStats \
     --kernel-name MatMulRegTilingKernel_v4 \
     --launch-skip 1 --launch-count 1 \
     ./matmul_reg_tiling_v4

==PROF== Connected to process 9387 (/content/matmul_reg_tiling_v4)
==PROF== Profiling "MatMulRegTilingKernel_v4": 0%....50%....100% - 1 pass
=== Register Tiling GEMM: v3 vs v4 ===
Matrix: 2048x2048x2048
Block tile: 128x128, Thread tile: 8x8, K tile: 8

v3 (PAD+swizzle):           5.456 ms  (3148.6 GFLOPS)
v4 (+ double buf + f4 st):  803.917 ms  (21.4 GFLOPS)
Speedup v4/v3: 0.01x
Max diff (v3 vs v4): 0.000000e+00
PASSED
==PROF== Disconnected from process 9387
[9387] matmul_reg_tiling_v4@127.0.0.1
  MatMulRegTilingKernel_v4(const float *, const float *, float *, int, int, int) (16, 16, 1)x(16, 16, 1), Context 1, Stream 7, Device 0, CC 7.5
    Section: Launch Statistics
    -------------------------------- --------------- -----------------
    Metric Name                          Metric Unit      Metric Value
    -------------------------------- --------------- -----------------
    Block Size                                                     256
    Function Cache Configuration        

In [10]:
# Scheduler + Warp Stall
!ncu --section SchedulerStats \
     --section WarpStateStats \
     --kernel-name MatMulRegTilingKernel_v4 \
     --launch-skip 1 --launch-count 1 \
     ./matmul_reg_tiling_v4

==PROF== Connected to process 9460 (/content/matmul_reg_tiling_v4)
==PROF== Profiling "MatMulRegTilingKernel_v4": 0%....50%....100% - 8 passes
=== Register Tiling GEMM: v3 vs v4 ===
Matrix: 2048x2048x2048
Block tile: 128x128, Thread tile: 8x8, K tile: 8

v3 (PAD+swizzle):           5.706 ms  (3010.6 GFLOPS)
v4 (+ double buf + f4 st):  2344.017 ms  (7.3 GFLOPS)
Speedup v4/v3: 0.00x
Max diff (v3 vs v4): 0.000000e+00
PASSED
==PROF== Disconnected from process 9460
[9460] matmul_reg_tiling_v4@127.0.0.1
  MatMulRegTilingKernel_v4(const float *, const float *, float *, int, int, int) (16, 16, 1)x(16, 16, 1), Context 1, Stream 7, Device 0, CC 7.5
    Section: Scheduler Statistics
    ---------------------------- ----------- ------------
    Metric Name                  Metric Unit Metric Value
    ---------------------------- ----------- ------------
    One or More Eligible                   %        50.70
    Issued Warp Per Scheduler                        0.51
    No Eligible              

In [11]:
# Speed of Light + Compute
!ncu --section SpeedOfLight \
     --section ComputeWorkloadAnalysis \
     --kernel-name MatMulRegTilingKernel_v4 \
     --launch-skip 1 --launch-count 1 \
     ./matmul_reg_tiling_v4

==PROF== Connected to process 9537 (/content/matmul_reg_tiling_v4)
==PROF== Profiling "MatMulRegTilingKernel_v4": 0%....50%....100% - 8 passes
=== Register Tiling GEMM: v3 vs v4 ===
Matrix: 2048x2048x2048
Block tile: 128x128, Thread tile: 8x8, K tile: 8

v3 (PAD+swizzle):           9.409 ms  (1825.9 GFLOPS)
v4 (+ double buf + f4 st):  2496.059 ms  (6.9 GFLOPS)
Speedup v4/v3: 0.00x
Max diff (v3 vs v4): 0.000000e+00
PASSED
==PROF== Disconnected from process 9537
[9537] matmul_reg_tiling_v4@127.0.0.1
  MatMulRegTilingKernel_v4(const float *, const float *, float *, int, int, int) (16, 16, 1)x(16, 16, 1), Context 1, Stream 7, Device 0, CC 7.5
    Section: GPU Speed Of Light Throughput
    ----------------------- ----------- ------------
    Metric Name             Metric Unit Metric Value
    ----------------------- ----------- ------------
    DRAM Frequency                  Ghz         5.00
    SM Frequency                    Mhz       585.00
    Elapsed Cycles                cycle    5,

In [12]:
# Source Counters
!ncu --section SourceCounters \
     --kernel-name MatMulRegTilingKernel_v4 \
     --launch-skip 1 --launch-count 1 \
     ./matmul_reg_tiling_v4

==PROF== Connected to process 9594 (/content/matmul_reg_tiling_v4)
==PROF== Profiling "MatMulRegTilingKernel_v4": 0%....50%....100% - 5 passes
=== Register Tiling GEMM: v3 vs v4 ===
Matrix: 2048x2048x2048
Block tile: 128x128, Thread tile: 8x8, K tile: 8

v3 (PAD+swizzle):           5.921 ms  (2901.4 GFLOPS)
v4 (+ double buf + f4 st):  2539.368 ms  (6.8 GFLOPS)
Speedup v4/v3: 0.00x
Max diff (v3 vs v4): 0.000000e+00
PASSED
==PROF== Disconnected from process 9594
[9594] matmul_reg_tiling_v4@127.0.0.1
  MatMulRegTilingKernel_v4(const float *, const float *, float *, int, int, int) (16, 16, 1)x(16, 16, 1), Context 1, Stream 7, Device 0, CC 7.5
    Section: Source Counters
    ------------------------- ----------- ------------
    Metric Name               Metric Unit Metric Value
    ------------------------- ----------- ------------
    Branch Instructions Ratio           %         0.03
    Branch Instructions              inst   12,060,672
    Branch Efficiency                   %        

In [13]:
# 导出完整报告
!ncu --set full \
     --kernel-name MatMulRegTilingKernel_v4 \
     --launch-skip 1 --launch-count 1 \
     -o reg_tiling_v4_profile \
     ./matmul_reg_tiling_v4

print("\n报告已保存为 reg_tiling_v4_profile.ncu-rep")

==PROF== Connected to process 9650 (/content/matmul_reg_tiling_v4)
==PROF== Profiling "MatMulRegTilingKernel_v4": 0%....50%....100% - 31 passes
=== Register Tiling GEMM: v3 vs v4 ===
Matrix: 2048x2048x2048
Block tile: 128x128, Thread tile: 8x8, K tile: 8

v3 (PAD+swizzle):           6.685 ms  (2569.8 GFLOPS)
v4 (+ double buf + f4 st):  8174.831 ms  (2.1 GFLOPS)
Speedup v4/v3: 0.00x
Max diff (v3 vs v4): 0.000000e+00
PASSED
==PROF== Disconnected from process 9650
==PROF== Report: /content/reg_tiling_v4_profile.ncu-rep

报告已保存为 reg_tiling_v4_profile.ncu-rep


In [14]:
from google.colab import files
files.download('reg_tiling_v4_profile.ncu-rep')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

---
## 对比检查清单

| 指标 | v1 (原版) | v3 (PAD+swizzle) | v4 (+ dbuf + f4st) | 变化 |
|------|-----------|-------------------|---------------------|------|
| Kernel 时间 (ms) | 8.378 | 5.713 | | |
| GFLOPS | 2050.5 | 3007.2 | | |
| Compute Throughput | 40.68% | 64.04% | | |
| Memory Throughput | 43.04% | 64.04% | | |
| IPC Active | 1.25 | 1.92 | | |
| Warp Cycles/Instruction | 12.22 | 7.74 | | |
| Eligible Warps/Scheduler | 0.42 | 1.08 | | |
| Global store util (bytes/sector) | 4.0/32 | 4.0/32 | | |
| Shared Mem/Block (KB) | 8.19 | 8.70 | | |
| Registers/Thread | 96 | 128 | | |
| Occupancy | 50% | 50% | | |